In [3]:
!pip install evaluate
!pip install transformers
!pip install datasets
!pip install seqeval
!pip install accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.4 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd

from datasets import load_dataset

from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from transformers import TrainingArguments
from transformers import Trainer

import evaluate

In [ ]:
dataset = load_dataset("universal_dependencies","en_ewt")

dataset

In [ ]:
label_list = dataset["train"].features["upos"].feature.names

label_list

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
label_list = dataset["train"].features["upos"].feature.names

id2label = {i:label for i,label in enumerate(label_list)}

label2id = {label:i for i,label in enumerate(label_list)}

In [ ]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["upos"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word_idx:

                label_ids.append(label[word_idx])

            else:

                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(

    output_dir="pos_model",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    evaluation_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="epoch"
)

In [ ]:
metric = evaluate.load("seqeval")

In [ ]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p,l) in zip(prediction,label) if l != -100]

        for prediction,label in zip(predictions,labels)
    ]

    true_labels = [

        [label_list[l] for (p,l) in zip(prediction,label) if l != -100]

        for prediction,label in zip(predictions,labels)

    ]

    results = metric.compute(

        predictions=true_predictions,

        references=true_labels

    )

    return {

        "precision": results["overall_precision"],

        "recall": results["overall_recall"],

        "f1": results["overall_f1"]

    }

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    tokenizer=tokenizer,

    compute_metrics=compute_metrics

)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
sentence = "John works at Google in California"

tokens = tokenizer(
    sentence.split(),
    is_split_into_words=True,
    return_tensors="pt"
)

outputs = model(**tokens)

predictions = outputs.logits.argmax(dim=2)

predicted_labels = [

    id2label[p.item()]

    for p in predictions[0]
]

print(sentence.split())

print(predicted_labels)

In [ ]:
'''
POS Tagging:

Identifies grammatical role
Example:

John → Noun
works → Verb

Chunking:

Identifies phrases
Example:

John → Noun Phrase
works at Google → Verb Phrase
'''

In [ ]:
'''
Difference table:

| POS             | Chunking         |
| --------------- | ---------------- |
| Word level      | Phrase level     |
| Grammar tagging | Phrase detection |
| Easier          | Harder           |
'''